# Assignment 4: Apache Spark Structured Streaming with Deduplication & Checkpointing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jits77111-source/DWBDA-Assignment-4-Spark-Structured-Streaming/blob/main/Assignment_4_Spark_Structured_Streaming.ipynb)

**Author:** Jit Sarkar  
**Course:** Data Warehousing, Big Data & Analytics (DWBDA)  
**GitHub Repository:** [DWBDA-Assignment-4-Spark-Structured-Streaming](https://github.com/jits77111-source/DWBDA-Assignment-4-Spark-Structured-Streaming)

---

### Objective
Demonstrate **Apache Spark Structured Streaming** for continuous real-time data ingestion from file streams, applying schema enforcement, **duplicate elimination using `dropDuplicates()`**, and **fault tolerance with state checkpointing**.

## 1. Install PySpark & Environment Setup

In [ ]:
!pip install -q pyspark

In [ ]:
import os, shutil, time, json
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

spark = SparkSession.builder \
    .appName("SparkStructuredStreamingAssignment") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Initialized Successfully!")

## 2. Directory Initialization for Stream & Checkpoints

- **`stream_input/`**: Directory where incoming JSON stream batches arrive.
- **`checkpoint/`**: Directory storing Write-Ahead Logs (WAL) and state offsets to ensure **exactly-once** processing.

In [ ]:
input_path = "/content/stream_input" if os.path.exists("/content") else os.path.abspath("stream_input")
checkpoint_path = "/content/checkpoint" if os.path.exists("/content") else os.path.abspath("checkpoint")

shutil.rmtree(input_path, ignore_errors=True)
shutil.rmtree(checkpoint_path, ignore_errors=True)

os.makedirs(input_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)
print(f"Stream input folder: {input_path}")
print(f"Checkpoint folder:   {checkpoint_path}")

## 3. Schema Definition & Streaming DataFrame Creation

In Spark Structured Streaming on file sources, schema must be explicitly defined upfront.

In [ ]:
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("speed_kmh", DoubleType(), True),
    StructField("fuel_pct", DoubleType(), True),
    StructField("status", StringType(), True)
])

# Create Streaming Read DataFrame
streaming_df = spark.readStream \
    .schema(schema) \
    .json(input_path)

print("Streaming DataFrame Source Attached!")
streaming_df.printSchema()

## 4. Apply Deduplication using `dropDuplicates()`

Networks and IoT message brokers often produce duplicate packets due to retries. `dropDuplicates(["event_id"])` ensures duplicate records are discarded.

In [ ]:
unique_stream_df = streaming_df.dropDuplicates(["event_id"])
print("Duplicate rejection policy applied on column: 'event_id'")

## 5. Start Streaming Query with Fault-Tolerant Checkpoint

We write streaming outputs to console sink with micro-batch trigger of 2 seconds.

In [ ]:
query = unique_stream_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="2 seconds") \
    .start()

print("Streaming Query Running! Status:", query.status)

## 6. Simulate Real-Time File Stream Ingestion (Micro-Batches)

In [ ]:
# Batch 1: Initial events
b1 = [
    {"event_id": "EVT_101", "device_id": "TRUCK_01", "timestamp": "2026-03-01 10:00:00", "speed_kmh": 72.5, "fuel_pct": 85.0, "status": "NORMAL"},
    {"event_id": "EVT_102", "device_id": "TRUCK_02", "timestamp": "2026-03-01 10:00:01", "speed_kmh": 95.0, "fuel_pct": 42.0, "status": "SPEEDING"}
]
with open(os.path.join(input_path, "data_batch_1.json"), "w") as f:
    for item in b1: f.write(json.dumps(item) + "\n")
print("Emitted Batch 1!")
time.sleep(4)

In [ ]:
# Batch 2: Includes duplicate EVT_102 and new EVT_103
b2 = [
    {"event_id": "EVT_102", "device_id": "TRUCK_02", "timestamp": "2026-03-01 10:00:01", "speed_kmh": 95.0, "fuel_pct": 42.0, "status": "SPEEDING"}, # DUPLICATE
    {"event_id": "EVT_103", "device_id": "TRUCK_03", "timestamp": "2026-03-01 10:00:05", "speed_kmh": 58.0, "fuel_pct": 17.5, "status": "LOW_FUEL"}
]
with open(os.path.join(input_path, "data_batch_2.json"), "w") as f:
    for item in b2: f.write(json.dumps(item) + "\n")
print("Emitted Batch 2 (with intentional duplicate)! Notice Spark ignores EVT_102.")
time.sleep(4)

In [ ]:
# Batch 3: Additional events
b3 = [
    {"event_id": "EVT_104", "device_id": "TRUCK_04", "timestamp": "2026-03-01 10:00:10", "speed_kmh": 88.0, "fuel_pct": 60.0, "status": "NORMAL"},
    {"event_id": "EVT_105", "device_id": "TRUCK_05", "timestamp": "2026-03-01 10:00:12", "speed_kmh": 102.5, "fuel_pct": 50.0, "status": "SPEEDING"}
]
with open(os.path.join(input_path, "data_batch_3.json"), "w") as f:
    for item in b3: f.write(json.dumps(item) + "\n")
print("Emitted Batch 3!")
time.sleep(4)

In [ ]:
# Process all pending records and terminate query
query.processAllAvailable()
query.stop()
print("Streaming Query Stopped Successfully.")
spark.stop()
print("Spark Session Terminated.")

## 7. Conclusion
- Successfully implemented PySpark Structured Streaming on real-time JSON files.
- Enforced a robust schema and eliminated duplicate events dynamically via `dropDuplicates()`.
- Configured fault-tolerant checkpointing with transaction logging.